# Drift Monitoring — Inference Monitoring Notebook

> 📊 **Note**: this notebook contains interactive output cells (Evidently reports, plotly charts). They render fully in JupyterLab; static viewers like the GitHub/GitLab renderer strip the JavaScript.

End-to-end drift monitoring for the deployed **bank-marketing** endpoint, run
entirely inside this notebook. All infrastructure (the `bank_marketing`
Athena tables, the endpoint's inference-capture path, the MLflow app) is
provisioned by the workshop CloudFormation stack (`scripts/deploy-workshop.sh`).
This notebook **creates no infrastructure** — no Lambda, SQS, EventBridge
schedule, SNS topic, or CloudWatch alarm. It computes drift and writes results
directly to Athena.

You focus on the parts that need a data scientist:

1. **Setup** — load config, confirm the monitoring tables exist
2. **Generate drifted data** *(dev/test only)* — fabricate traffic that should trip the detector
3. **Send predictions** — invoke the endpoint so there's something to monitor
4. **Apply ground truth** — simulate labels for model-drift metrics
5. **Detect drift** — the core: data drift + model drift with Evidently, written to Athena and logged to MLflow
6. **Simulate a drift trend** — repeat 3–5 in a loop of batches so the governance dashboard has a trend to chart

> **Rule**: never send predictions (Section 3) *after* running the ground-truth simulator (Section 4) — those rows would have NULL ground_truth and skew model-drift metrics. The cell order enforces this.


## Prerequisites

This notebook assumes the following already exist (all created by
`scripts/deploy-workshop.sh` plus the earlier labs). If any are missing,
create them before continuing:

| Prerequisite | How it's created |
|---|---|
| SageMaker domain, S3 data bucket, MLflow app, `bank_marketing` Athena tables (`training_data`, `evaluation_data`, `inference_responses`, `monitoring_responses`, `ground_truth_updates`), inference-capture path | Workshop stack (`scripts/deploy-workshop.sh` → `templates/`) |
| A trained + **Approved** model in the `bank-prediction-XGBoostModel` model package group | Lab 3A / 3C |
| A live `bank-marketing-<timestamp>` endpoint logging to `inference_responses` | Lab 3A |

The `.env` file (populated by the domain lifecycle script) carries `DATA_S3_BUCKET`
and `AWS_DEFAULT_REGION`; everything else resolves from the workshop stack
outputs via `src/config/config.py`. The endpoint is discovered live (the most
recent InService `bank-marketing-*` endpoint) — nothing to hand-edit.


## 1. Setup

In [ ]:
# Load the shared .env (written by lab0-setup/setup.ipynb), install this lab's
# package so `import src` resolves, and clear stale src modules from the kernel.
from src.utils.notebook_env import bootstrap
env = bootstrap(install_package=True)

In [ ]:
import logging
from IPython.display import display

from src.config.config import (
    CSV_DRIFTED_DATA, GROUND_TRUTH_SIM_ACCURACY,
    GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT, GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
)
from src.drift_monitoring.monitoring_run import MonitoringSession, print_per_column_drift

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Resolves the live Lab 3A endpoint (most recent InService bank-marketing-*,
# unless endpoint.name is pinned in config.yaml) and the drift window.
session = MonitoringSession()
session.describe()

### 1.1 Monitoring tables (provisioned by the workshop stack)

The Athena tables this notebook reads and writes — `inference_responses`,
`monitoring_responses`, and `ground_truth_updates` — are all created by the
**workshop CloudFormation stack** (`deploy-workshop.sh` →
`templates/4-inference-capture.yaml`), in the `bank_marketing` database.

This notebook **creates no infrastructure**: it only writes rows into those
tables. There is no drift-monitoring Lambda, SQS queue, EventBridge schedule,
SNS topic, or CloudWatch alarm — drift is computed and persisted inline below.


In [ ]:
# Read-only: confirm the monitoring tables exist. Creates nothing — if one is
# missing, redeploy the workshop stack (scripts/deploy-workshop.sh).
missing_tables = session.check_tables()

### 1.2 What this notebook does

Sends predictions to the live endpoint, applies simulated ground truth,
computes **data drift** and **model drift** with Evidently, writes the run
to `monitoring_responses`, and logs the Evidently reports + metrics to
**MLflow**. The `monitoring_responses` / `inference_responses` /
`ground_truth_updates` tables then back the **QuickSight** governance
dashboard (Lab 5e).


## 2. Generate Drifted Test Data *(dev/test only)*

Fabricate a drifted CSV the detector should flag, so Section 5 has something to
find. In production you'd skip this entirely — real traffic supplies the drift.
Drift parameters live in `src/config/config.yaml` → `drift_generation.default_drift`.

In [ ]:
# Writes data/drifted_data.csv. Drift amounts: config.yaml → drift_generation.default_drift
from src.drift_monitoring.generate_drift_dataset import generate_drifted_dataset

generate_drifted_dataset()

## 3. Send Predictions to the Endpoint

Every cell here invokes the deployed endpoint. The custom handler auto-logs each
prediction to Athena via SQS → Lambda → `inference_responses` (batches of 10 or
every 30s). Most invocations send intentionally-drifted samples from Section 2;
a small sanity check confirms the endpoint is alive.

### 3.1 Endpoint helper

Every call in this notebook goes through `session.invoke()`
(`src/drift_monitoring/monitoring_run.py`). It posts one JSON row to the
endpoint and returns the prediction and P(positive). The custom handler logs
each request to Athena on its own; nothing here writes `inference_responses`.

### 3.2 Sanity check: one hard-coded payload

In [ ]:
# One non-drifted prediction on the training scale (values in
# monitoring_run.SANITY_DEFAULTS) to confirm the endpoint is alive.
sanity = session.sanity_check()

### 3.3 Bulk drifted inferences

In [ ]:
# Send drifted rows from Section 2, then wait until they land in Athena
# (polls inference_responses instead of sleeping a fixed time).
NUM_BULK_TESTS = 101
bulk_stats = session.send_drifted_csv(CSV_DRIFTED_DATA, n=NUM_BULK_TESTS)

### 3.4 Verify predictions landed in Athena

In [ ]:
counts = session.prediction_counts()

## 4. Apply Ground Truth

In production, labels arrive asynchronously (e.g. from investigations, manual
review, or delayed outcome observation). For dev/test we simulate them here so
model-drift metrics have something to score.

**Do not re-run Section 3 after this point** — newly logged predictions would
have NULL ground_truth and wouldn't contribute to model drift until the
simulator runs again.

### 4.1 Simulator parameters

In [ ]:
# Ground truth simulation parameters from src/config/config.yaml → ground_truth_simulation.
# Only two knobs reduce simulated accuracy:
#   effective_accuracy = max(0.5, base_accuracy
#                                 - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT
#                                 - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'Base accuracy        : {GROUND_TRUTH_SIM_ACCURACY:.2f}')
print(f'Feature-drift impact : -{GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT:.2f}')
print(f'Model-drift magnitude: -{GROUND_TRUTH_SIM_MODEL_DRIFT_MAG:.2f}')
effective = max(0.5, GROUND_TRUTH_SIM_ACCURACY - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'\nEffective accuracy   : {effective:.2f}  → ~{int((1 - effective) * 100)}% of labels will be flipped')


### 4.2 Generate and apply simulated labels

In [ ]:
# Simulate labels for every unlabelled prediction (GroundTruthSimulator) and
# MERGE them into inference_responses (GroundTruthUpdater).
gt_stats = session.apply_ground_truth(
    accuracy=GROUND_TRUTH_SIM_ACCURACY,
    feature_drift_impact=GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    model_drift_magnitude=GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
    seed=42,
)

### 4.3 Verify coverage

In [ ]:
coverage = session.coverage()

## 5. Detect Drift

The core of the notebook. Two independent checks, both traceable to the exact
model + data snapshot that produced the predictions (lineage pinned in
`baseline.json` on the registered ModelPackage):

- **5.2 Data drift** — Evidently `DataDriftPreset` (auto-selects KS / Chi-square
  for small samples, Wasserstein / Jensen-Shannon for n ≥ 1000). Baseline =
  `training_data`; current = `inference_responses` in the run's window.
- **5.3 Model drift** — classification metrics plus Evidently
  `ClassificationPreset`. Baseline = `evaluation_data`; current = labelled
  `inference_responses` in the run's window.

**One Evidently invocation, two consumers.** The same output is written to
(a) the `monitoring_responses` Athena table and (b) the MLflow experiment
(metrics + Evidently HTML artifact). Section 6 runs the *same*
`MonitoringSession` methods in a loop, so the numbers in this notebook, in
QuickSight, and in MLflow are always the same Evidently output. No custom drift
math is applied anywhere between compute and dashboard.

### 5.0 Run setup — `monitoring_run_id`, window, baseline lineage

Each pass produces one `monitoring_run_id`, reused by 5.2 / 5.3 / 5.4. The
"current" window comes from `config.yaml` → `drift_window`:

| `mode` | Window |
|---|---|
| `since_last_run` (default) | Predictions after the previous monitoring run, so re-running measures *new* drift, not all cumulative traffic. The first-ever run uses the last 7 days |
| `rolling` | Predictions in the last `minutes` before the run |

In [ ]:
# Allocates monitoring_run_id, resolves the window, prints baseline lineage.
run = session.start_run()

### 5.1 Overall classification metrics

`ModelPerformanceMonitor` computes precision/recall/F1/ROC-AUC against
`ground_truth` — the "current" half of the model-drift comparison, printed
inline before the detailed Evidently report.

In [ ]:
# Daily windows over the last 7 days, each judged against training-time ROC-AUC.
perf_report = session.performance_report(days=7)

### 5.2 Data drift

Per-feature drift test (Evidently `DataDriftPreset`). Baseline = `training_data`
(the distribution the model was trained on); current = `inference_responses`
within the window resolved in 5.0. Features that cannot be compared due to type
mismatches (e.g., string in training vs. encoded float in inference) are excluded.

**Which test runs?** Evidently auto-selects per column based on sample size:

| Column | Sample size | Test | Direction |
|---|---|---|---|
| Numeric | n < 1000 | Kolmogorov–Smirnov | p-value, lower = drift |
| Numeric | n ≥ 1000 | Wasserstein distance | distance, higher = drift |
| Categorical | n < 1000 | Chi-square | p-value, lower = drift |
| Categorical | n ≥ 1000 | Jensen-Shannon distance | distance, higher = drift |

Because raw `drift_score` has opposite drift directions across tests, the
notebook reports `drift_magnitude` (× past threshold) as the primary metric —
`1.0` = at threshold, `> 1.0` = drifted, higher = worse, regardless of test.

In [ ]:
drift_results = session.data_drift(run)

In [ ]:
# Interactive Evidently DataDriftPreset report.
if drift_results:
    display(drift_results['snapshot'])

In [ ]:
# Every feature, most drifted first. drift_magnitude is test-agnostic:
# 1.0 = at threshold, > 1.0 = drifted.
if drift_results:
    print_per_column_drift(drift_results)

### 5.3 Model drift

Classification-metric drift (Evidently `ClassificationPreset`). Baseline
`(target, prediction)` pairs come from `evaluation_data` — re-scored here with
the registered artifact, because that table's own `prediction` column is NULL;
current `(ground_truth, prediction)` from labeled `inference_responses`.

**Where the degradation comes from.** Model drift is a *label* phenomenon, so it
needs ground truth — and in this lab the ground truth is simulated (Section 4).
The simulator sets `actual = prediction` and then flips `1 - effective_accuracy`
of the labels at random, so the measured ROC-AUC lands well below the training
baseline (~0.78 vs ~0.96 at the default 0.85 accuracy) and the monitor alerts.
Turning up `GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT` / `GROUND_TRUTH_SIM_MODEL_DRIFT_MAG`
deepens it. Note this means the feature drift injected in Section 2 is **not**
what causes the degradation — data drift and model drift are simulated
independently here, which is also how the upstream reference sample does it.

If model drift is skipped, the cell prints why (too few labeled rows, or one
class only) — re-run Section 3.3 then Section 4.2.

In [ ]:
# Widens to the last 30 days if this run's window is too small to score.
metrics, model_report = session.model_drift(run, widen_days=30)
if model_report:
    display(model_report['snapshot'])

### 5.4 Log results to MLflow and Athena

Two writes keyed on the same `monitoring_run_id`: (1) an MLflow run with all
drift metrics + the Evidently HTML reports as artifacts; (2) one row in
`monitoring_responses` (with model-package ARN + snapshot id) plus a backfill
tagging the inference rows this run measured, so QuickSight can join them.

The model-drift columns (`current_roc_auc`, `accuracy`, …) describe this run's
window only, and `model_drift_detected` is `current_roc_auc` below
`drift_thresholds.min_roc_auc`. The 7-day report in 5.1 is for reading only and
is not written.

In [ ]:
row = session.finish_run(run, drift_results, metrics, model_report)

## 6. Simulate a drift trend (multiple batches)

Sections 2–5 produce **one** monitoring run, which is one point on each Lab 5E
chart. This section repeats that pass in a loop, calling the same `MonitoringSession`
methods, so the governance dashboard has a trend to show for **data drift**,
**feature drift**, and **model drift** (`src/drift_monitoring/batch_simulation.py`).

Each batch in `src/config/config.yaml` → `drift_simulation.batches` sends
`rows_per_batch` predictions, simulates their labels, computes drift over the
**drift window**, and writes one row to `monitoring_responses`. The tables and
their columns are the same as in Section 5.4. Nothing new is created.

| Knob | Where | Effect |
|---|---|---|
| `intensity` | per batch | Scales `drift_generation.default_drift` (0 = clean, 1 = as written) → **data / feature drift** |
| `gt_accuracy` | per batch | Simulated label accuracy → **model drift** (lower = worse ROC-AUC) |
| `drift_window.mode` | `drift_window` | `since_last_run`: each run measures only new traffic. `rolling`: the last `minutes` before each run (windows overlap) |
| `drift_window.minutes` | `drift_window` | Width of the `rolling` window |

Each batch takes about 1–3 minutes: invocations, then waiting for the
SQS → Lambda capture, the ground-truth MERGE, and the Athena writes. Batches run
back to back, so they are minutes apart. The Lab 5E trend visuals therefore
group runs by **minute** (`QUICKSIGHT_TREND_GRANULARITY` in
`src/config/config.py`). Set that to `DAY` for a once-a-day production schedule.

In [ ]:
from src.config.config import DRIFT_SIM_BATCHES
from src.drift_monitoring.batch_simulation import BatchSimulation, plot_trend, print_schedule

# Try another window without editing config.yaml, e.g.
#   session = MonitoringSession(window_mode='rolling', window_minutes=15)
BATCHES = DRIFT_SIM_BATCHES
print_schedule(BATCHES)
print(f'Drift window: {session.window_label()}')
sim = BatchSimulation(session)

In [ ]:
# One monitoring_responses row per batch. Re-running adds more runs; with
# since_last_run each new run measures only traffic sent after the previous one.
trend_df = sim.run(BATCHES)
trend_df

### 6.1 Preview the trend (what Lab 5E will chart)

Reads every run for this endpoint back from `monitoring_responses`, not just
this loop's runs. It unpacks `per_feature_drift_scores` the same way Lab 5E's
`feature_drift_detail` view does, so these are the values QuickSight charts.

In [ ]:
trend_runs = plot_trend(session)

## Teardown

This notebook created no infrastructure, so there is nothing here to tear down —
the drifted CSV and the Evidently HTML reports are local files, and the drift
results are rows in the `bank_marketing` Athena tables that the workshop stack
owns.

Two things *are* still costing money after this lab, and neither belongs to this
notebook:

- **The endpoint.** Lab 3A's final cell deletes the endpoint, its config, and the
  model. Run it once you have finished every Lab 5 notebook — 5A through 5F all
  need a live endpoint.
- **The workshop stack.** When you are completely done:
  ```bash
  aws cloudformation delete-stack --stack-name bank-marketing-prediction-workshop
  ```
  The S3 buckets are retained on purpose; `scripts/delete_s3_buckets.py` empties
  and removes them once you no longer need the data.

---